In [1]:
import subprocess
subprocess.run(["pip", "install", "bert-score", "--trusted-host", "pypi.org", "--trusted-host", "files.pythonhosted.org"], check=True)

CompletedProcess(args=['pip', 'install', 'bert-score', '--trusted-host', 'pypi.org', '--trusted-host', 'files.pythonhosted.org'], returncode=0)

In [2]:
# === RUN THIS CELL FIRST: SSL Fix ===
import ssl
import os

ssl._create_default_https_context = ssl._create_unverified_context
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '0'

import requests
old_request = requests.Session.request
def new_request(self, *args, **kwargs):
    kwargs['verify'] = False
    return old_request(self, *args, **kwargs)
requests.Session.request = new_request

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print('SSL verification disabled')

SSL verification disabled


In [3]:
# ============================================================
# LLM Selection: Key Principles Accuracy — All 3 Models
# Evaluates Claude Sonnet 4, DeepSeek-Chat, GPT-4o
# against manual annotations by Chloe
# Metric: Hybrid BERTScore (50%) + Term Overlap (50%)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import torch
from bert_score import score
import re
import random

print('=' * 80)
print('LLM SELECTION: KEY PRINCIPLES ACCURACY — ALL 3 MODELS')
print('=' * 80)
print(f'Started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

# ── CONFIG ────────────────────────────────────────────────────
NUM_ITERATIONS = 50
LEGAL_BERT_MODEL = '/Users/shannon/legal-bert'
NUM_LAYERS = 9
TERM_WEIGHT = 0.5
BERT_WEIGHT = 0.5
RANDOM_SEED = 42

# Base directory — update this to point to your LLM Selection folder
BASE_DIR = Path.home() / 'Desktop' / 'LLM Selection'

# Model configurations: (display name, folder name, column name in xlsx)
MODELS = [
    ('Claude Sonnet 4',  'Claude Sonnet 4_individual_key_principle_extraction_results',  'KeyPrinciple_CLAUDE'),
    ('DeepSeek-Chat',    'DeepSeek-Chat_individual_key_principle_extraction_results',    'KeyPrinciple_DEEPSEEK'),
    ('GPT-4o',          'GPT-4o_individual_key_principle_extraction_results',            'KeyPrinciple_CHATGPT'),
]

# 3 Groups (broad)
CASE_GROUPS_3 = {
    'GROUP 1: Corporate, Commercial & Insolvency': [2, 22, 24, 8, 15, 17, 10, 16, 14, 19, 18, 20],
    'GROUP 2: Public Law & Regulation':            [5, 12, 4, 11, 7, 9],
    'GROUP 3: Employment, Personal & Civil Disputes': [6, 13, 21, 23, 25, 3, 1]
}

OUTPUT_FILE = Path('LLM_Selection_KeyPrinciples_Results.xlsx')

# ── HELPER FUNCTIONS ──────────────────────────────────────────

def extract_legal_terms(text, min_length=3):
    if not text or pd.isna(text):
        return set()
    text = str(text).lower()
    stopwords = {'the','a','an','and','or','but','in','on','at','to','for',
                 'of','with','by','from','as','is','was','be','been','are',
                 'has','have','had','that','which','who','whom','this','these',
                 'those','it','its'}
    words = re.findall(r'\b[a-z]+\b', text)
    terms = set()
    for word in words:
        if len(word) >= min_length and word not in stopwords:
            terms.add(word)
    words_list = [w for w in words if w not in stopwords]
    for i in range(len(words_list) - 1):
        terms.add(f'{words_list[i]} {words_list[i+1]}')
    for i in range(len(words_list) - 2):
        trigram = f'{words_list[i]} {words_list[i+1]} {words_list[i+2]}'
        if any(w in trigram for w in ['natural justice','breach','duty','reasonable','tribunal']):
            terms.add(trigram)
    return terms

def calculate_term_overlap(text1, text2):
    if not text1 or not text2 or pd.isna(text1) or pd.isna(text2):
        return 0.0
    terms1 = extract_legal_terms(text1)
    terms2 = extract_legal_terms(text2)
    if not terms1 or not terms2:
        return 0.0
    intersection = terms1.intersection(terms2)
    union = terms1.union(terms2)
    jaccard = len(intersection) / len(union) if union else 0.0
    key_phrases = [
        'natural justice','breach of duty','reasonable care',
        'contractual obligation','statutory interpretation',
        'constitutional challenge','judicial review','abuse of process',
        'burden of proof','standard of proof','procedural fairness',
        'substantive fairness','legitimate expectation'
    ]
    bonus = sum(0.15 for p in key_phrases if p in str(text1).lower() and p in str(text2).lower())
    return min(jaccard + bonus, 1.0)

def sample_one_iteration(input_dir, col_name, case_groups, iteration_num, seed):
    random.seed(seed)
    np.random.seed(seed)
    sampled_data = []
    for group_name, case_list in case_groups.items():
        shuffled = case_list.copy()
        random.shuffle(shuffled)
        for case_id in shuffled:
            case_file = input_dir / f'Case_{case_id}_KeyPrinciple_Extraction.xlsx'
            if not case_file.exists():
                continue
            try:
                df = pd.read_excel(case_file)
                df_valid = df[
                    df['KeyPrinciple_Annotated'].notna() &
                    df[col_name].notna() &
                    (df['KeyPrinciple_Annotated'].astype(str).str.strip() != '') &
                    (df[col_name].astype(str).str.strip() != '')
                ]
                if len(df_valid) == 0:
                    continue
                row = df_valid.sample(n=1, random_state=np.random.randint(0, 100000)).copy()
                row['Source_Case']  = case_id
                row['Source_Group'] = group_name
                row['Iteration']    = iteration_num
                sampled_data.append(row)
                break
            except Exception:
                continue
    if len(sampled_data) == len(case_groups):
        return pd.concat(sampled_data, ignore_index=True)
    return None

def calculate_accuracy(df_sample, col_name):
    annotated = [str(x) if pd.notna(x) else '' for x in df_sample['KeyPrinciple_Annotated'].tolist()]
    predicted = [str(x) if pd.notna(x) else '' for x in df_sample[col_name].tolist()]
    n = len(annotated)
    term_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            term_matrix[i, j] = calculate_term_overlap(predicted[i], annotated[j])
    f1_matrix = np.zeros((n, n))
    for col_idx in range(n):
        refs = [annotated[col_idx]] * n
        P, R, F1 = score(
            predicted, refs,
            model_type=LEGAL_BERT_MODEL, num_layers=NUM_LAYERS,
            verbose=False,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        f1_matrix[:, col_idx] = F1.numpy()
    hybrid = (TERM_WEIGHT * term_matrix) + (BERT_WEIGHT * f1_matrix)
    correct = sum(np.argmax(hybrid[:, j]) == j for j in range(n))
    return correct, n

# ── MAIN EVALUATION LOOP ──────────────────────────────────────

print(f'\nDevice: {"CUDA" if torch.cuda.is_available() else "CPU"}')
print(f'Weights: {int(TERM_WEIGHT*100)}% Term + {int(BERT_WEIGHT*100)}% BERT')
print(f'Iterations per model: {NUM_ITERATIONS}')

all_results  = []   # per-iteration details
summary_rows = []   # one row per model

for model_name, folder_name, col_name in MODELS:
    input_dir = BASE_DIR / folder_name
    print(f'\n{"="*80}')
    print(f'MODEL: {model_name}')
    print(f'Folder: {input_dir}')
    print(f'Column: {col_name}')
    print(f'{"="*80}')

    if not input_dir.exists():
        print(f'ERROR: Folder not found — {input_dir}')
        print('Please update BASE_DIR at the top of this notebook.')
        continue

    total_correct = 0
    total_n       = 0
    perfect       = 0

    for iteration in range(1, NUM_ITERATIONS + 1):
        seed = RANDOM_SEED + iteration
        print(f'  Iteration {iteration}/{NUM_ITERATIONS}...', end=' ')
        df_sample = sample_one_iteration(input_dir, col_name, CASE_GROUPS_3, iteration, seed)
        if df_sample is None:
            print('Failed')
            continue
        correct, n = calculate_accuracy(df_sample, col_name)
        total_correct += correct
        total_n       += n
        if correct == n:
            perfect += 1
        acc = 100 * correct / n if n > 0 else 0
        print(f'{correct}/{n} ({acc:.1f}%)')
        all_results.append({
            'Model':       model_name,
            'Iteration':   iteration,
            'Correct':     correct,
            'Total':       n,
            'Accuracy_%':  acc,
            'Perfect':     'Yes' if correct == n else 'No'
        })

    overall_acc = 100 * total_correct / total_n if total_n > 0 else 0
    print(f'\n  >> {model_name}: {total_correct}/{total_n} = {overall_acc:.1f}% accuracy')
    summary_rows.append({
        'Model':             model_name,
        'Total_Correct':     total_correct,
        'Total_Samples':     total_n,
        'Accuracy_%':        round(overall_acc, 1),
        'Perfect_Iterations': perfect,
        'Waterfall_Rate_%':  round(100 * perfect / NUM_ITERATIONS, 1),
    })

# ── FINAL SUMMARY ─────────────────────────────────────────────

print(f'\n{"="*80}')
print('FINAL SUMMARY')
print(f'{"="*80}')
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── SAVE ──────────────────────────────────────────────────────

details_df = pd.DataFrame(all_results)
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary',          index=False)
    details_df.to_excel(writer, sheet_name='Iteration_Details', index=False)

print(f'\nSaved to {OUTPUT_FILE}')
print(f'Completed at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

/Users/shannon/miniconda3/envs/minienv/lib/python3.11/site-packages/sklearn/utils/__init__.py:16: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.3)
  from scipy.sparse import issparse


LLM SELECTION: KEY PRINCIPLES ACCURACY — ALL 3 MODELS
Started at: 2026-03-11 20:24:27

Device: CPU
Weights: 50% Term + 50% BERT
Iterations per model: 50

MODEL: Claude Sonnet 4
Folder: /Users/shannon/Desktop/LLM Selection/Claude Sonnet 4_individual_key_principle_extraction_results
Column: KeyPrinciple_CLAUDE
  Iteration 1/50... 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

BertModel LOAD REPORT from: /Users/shannon/legal-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.pred

2/3 (66.7%)

  >> GPT-4o: 127/150 = 84.7% accuracy

FINAL SUMMARY
          Model  Total_Correct  Total_Samples  Accuracy_%  Perfect_Iterations  Waterfall_Rate_%
Claude Sonnet 4            137            150        91.3                  38              76.0
  DeepSeek-Chat            130            150        86.7                  33              66.0
         GPT-4o            127            150        84.7                  29              58.0

Saved to LLM_Selection_KeyPrinciples_Results.xlsx
Completed at: 2026-03-11 20:30:57
